In [256]:
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

In [257]:
config = {
    'BATCH_SIZE':64,
}

In [258]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Multi-Layer Perceptron Implementation

## Model Creation

In [259]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(41250,200)
        self.l2 = nn.Linear(200, 30)
        self.out = nn.Linear(30,3)
        self.bn = nn.BatchNorm1d(200)
        self.relu = nn.ReLU()

    def forward(self,x):
        x = self.l1(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.l2(x)
        x = self.relu(x)
        x = self.out(x)
        return x

In [260]:
model = MLP()
# model = model.to("cuda" if torch.cuda.is_available() else "cpu")

sum(p.numel() for p in model.parameters())

8256723

## Dataset Preparation

In [261]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms
import PIL

class GameplayDataset(Dataset):
    def __init__(self,train=True):
        super().__init__()
        #Input_text
        with open(r'CNN_Training_Data\inputs.txt','r') as f:
            self.input_data = [int(a) for a in list(f.read())]
        if train:
            self.len = int(len(self.input_data) * 0.8)
        else:
            self.len = int(len(self.input_data) * 0.2)
            self.last = int(len(self.input_data) * 0.8)

        
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((150,275)),
            transforms.ToTensor()
            ])

        self.train = train


    def __getitem__(self,idx):
        if not self.train:
            idx = self.last + idx
        #Images
        img_path = f"CNN_Training_Data/IMAGES/{idx}.png"
        img = cv2.imread(img_path)
        bw_img = (cv2.cvtColor(img,cv2.COLOR_BGR2GRAY) > 127).astype(float)
        bw_img = self.transform(bw_img)
        bw_img = bw_img.flatten()
        return bw_img,self.input_data[idx]

    def __len__(self):
        return self.len



In [262]:
train_dataset = GameplayDataset(True)
val_dataset = GameplayDataset(False)

train_dataloader = DataLoader(dataset=train_dataset,
                              batch_size=config['BATCH_SIZE'],
                              shuffle=True,
                              num_workers=0,
                              )

val_dataloader = DataLoader(dataset=val_dataset,
                            batch_size=config['BATCH_SIZE'],
                            shuffle=False,
                            num_workers=0)

In [263]:
from tqdm.notebook import tqdm
optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)

def train(epoch):
    model.train()
    train_loss = 0
    for batch_idx, (input,output) in tqdm(enumerate(train_dataloader),total=len(train_dataloader)):
        optimizer.zero_grad()
        y_pred = model(input)
        loss = F.cross_entropy(y_pred,output)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
        

    print(f'Epoch {epoch}: Average loss: {train_loss / (len(train_dataloader)):.4f}') 



In [264]:
# # Evaluate

# model.eval()
# predictions = []
# truths = []

# for x,y in val_dataloader:
#     pred = torch.argmax(model(x),dim=-1).tolist()
#     predictions.extend(pred)
#     truths.extend(y.tolist())
  

In [265]:
next(model.parameters()).device

device(type='cpu')

In [266]:
from sklearn.metrics import accuracy_score, f1_score

print(f"The accuracy of the model is {accuracy_score(predictions,truths)}")
print(f"The F1 score of the model is {f1_score(predictions,truths,average='macro')}")

The accuracy of the model is 0.28958333333333336
The F1 score of the model is 0.14970382337102853


***
After 2 epochs we get:
Accuracy = ~0.6
F1 = ~0.47

# CNN Implementation
## Model Definintion

In [267]:
class CNN(nn.Module):
    def __init__(self,in_channels = 1,num_classes=3,channels = 32,final_pool=1):
        super().__init__()
        conv1_in_channnels = in_channels
        conv1_out_channels = channels
        conv2_in_channels = channels
        conv2_out_channels = channels * 2
        #Convolution layers
        self.conv1 = nn.Conv2d(in_channels=conv1_in_channnels,
                               out_channels=conv1_out_channels,
                               kernel_size=5,
                               stride=1)

        self.conv2 = nn.Conv2d(in_channels=conv2_in_channels,
                               out_channels=conv2_out_channels,
                               kernel_size=5,
                               stride=1)

        #Max=Pooling layers
        self.max1 = nn.MaxPool2d(kernel_size=2)
        self.max2 = nn.AdaptiveMaxPool2d((final_pool,final_pool))

        #Fully Connected layers
        self.fc1 = nn.Linear(conv2_out_channels*final_pool*final_pool,32)
        self.fc2 = nn.Linear(32,num_classes)

        #Activation Layer
        self.relu =nn.ReLU()

        #Regularisation
        self.bn1 = nn.BatchNorm2d(num_features=conv1_out_channels)
        self.bn2 = nn.BatchNorm2d(num_features=conv2_out_channels)


    def forward(self,x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.max1(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.max2(x)
        x = torch.flatten(x,start_dim =1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x



## Prepare Dataset for CNN

In [268]:
class GameplayDatasetCNN(Dataset):
    def __init__(self,train=True):
        super().__init__()
        #Input_text
        with open(r'CNN_Training_Data\inputs.txt','r') as f:
            self.input_data = [int(a) for a in list(f.read())]
        if train:
            self.len = int(len(self.input_data) * 0.8)
        else:
            self.len = int(len(self.input_data) * 0.2)
            self.last = int(len(self.input_data) * 0.8)

        
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((216,216)),
            transforms.ToTensor()
            ])

        self.train = train


    def __getitem__(self,idx):
        if not self.train:
            idx = self.last + idx
        #Images
        img_path = f"CNN_Training_Data/IMAGES/{idx}.png"
        img = Image.open(img_path)
        bw_img = self.transform(img)
        return bw_img,self.input_data[idx]

    def __len__(self):
        return self.len

In [269]:
train_dataset_cnn = GameplayDatasetCNN(True)
val_dataset_cnn = GameplayDatasetCNN(False)

train_dataloader_cnn = DataLoader(dataset=train_dataset_cnn,
                              batch_size=config['BATCH_SIZE'],
                              shuffle=True,
                              num_workers=0,
                              )

val_dataloader_cnn = DataLoader(dataset=val_dataset_cnn,
                            batch_size=config['BATCH_SIZE'],
                            shuffle=False,
                            num_workers=0)

In [270]:
cnn_model = CNN()
sum(p.numel() for p in cnn_model.parameters())

54467

In [271]:
cnn_model = cnn_model.to(device)

In [275]:
optimizer = torch.optim.Adam(cnn_model.parameters(),lr=1e-3)

def train(epoch):
    cnn_model.train()
    train_loss = 0
    for batch_idx, (input,output) in tqdm(enumerate(train_dataloader_cnn),total=len(train_dataloader_cnn)):
        optimizer.zero_grad()
        input = input.to(device)
        output = output.to(device)
        y_pred = cnn_model(input)
        loss = F.cross_entropy(y_pred,output)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
        

    print(f'Epoch {epoch}: Average loss: {train_loss / (len(train_dataloader)):.4f}') 
    print("Evaluation")
    model.eval()
    predictions = []
    truths = []

    for x,y in tqdm(val_dataloader_cnn,total=len(val_dataloader_cnn)):
        x = x.to(device)
        y = y.to(device)
        pred = torch.argmax(cnn_model(x),dim=-1).tolist()
        predictions.extend(pred)
        truths.extend(y.tolist())

    print(f"Validation Accuracy : {accuracy_score(truths,predictions)} | Validation F1 Score : {f1_score(truths,predictions,average='macro')}")

for i in range(1,4):
    train(i)

  0%|          | 0/189 [00:00<?, ?it/s]

Epoch 1: Average loss: 0.2184
Evaluation


  0%|          | 0/48 [00:00<?, ?it/s]

Validation Accuracy : 0.8908427339084274 | Validation F1 Score : 0.891339913943488


  0%|          | 0/189 [00:00<?, ?it/s]

...

Validation Accuracy : 0.9190444591904446 | Validation F1 Score : 0.9181627014986992


  0%|          | 0/189 [00:00<?, ?it/s]

Epoch 3: Average loss: 0.1988
Evaluation


  0%|          | 0/48 [00:00<?, ?it/s]

Validation Accuracy : 0.9253483742534837 | Validation F1 Score : 0.9243394535982276


In [276]:
cnn_model = CNN(1,num_classes=3,channels=16,final_pool=8)
cnn_model = cnn_model.to(device)

In [277]:
optimizer = torch.optim.Adam(cnn_model.parameters(),lr=1e-3)

def train(epoch):
    cnn_model.train()
    train_loss = 0
    for batch_idx, (input,output) in tqdm(enumerate(train_dataloader_cnn),total=len(train_dataloader_cnn)):
        optimizer.zero_grad()
        input = input.to(device)
        output = output.to(device)
        y_pred = cnn_model(input)
        loss = F.cross_entropy(y_pred,output)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
        

    print(f'Epoch {epoch}: Average loss: {train_loss / (len(train_dataloader)):.4f}') 
    print("Evaluation")
    model.eval()
    predictions = []
    truths = []

    for x,y in tqdm(val_dataloader_cnn,total=len(val_dataloader_cnn)):
        x = x.to(device)
        y = y.to(device)
        pred = torch.argmax(cnn_model(x),dim=-1).tolist()
        predictions.extend(pred)
        truths.extend(y.tolist())

    print(f"Validation Accuracy : {accuracy_score(truths,predictions)} | Validation F1 Score : {f1_score(truths,predictions,average='macro')}")   
    

for i in range(1,4):
    train(i)

  0%|          | 0/189 [00:00<?, ?it/s]

Epoch 1: Average loss: 0.2703
Evaluation


  0%|          | 0/48 [00:00<?, ?it/s]

Validation Accuracy : 0.8988055739880557 | Validation F1 Score : 0.8928470849681694


  0%|          | 0/189 [00:00<?, ?it/s]

...

Validation Accuracy : 0.9382879893828799 | Validation F1 Score : 0.9365567395898765


  0%|          | 0/189 [00:00<?, ?it/s]

Epoch 3: Average loss: 0.1377
Evaluation


  0%|          | 0/48 [00:00<?, ?it/s]

Validation Accuracy : 0.9442601194426012 | Validation F1 Score : 0.9430785623860496
